# PR0303: Obtención de datos de una API REST

In [1]:
!pip install requests

## 1.- Conexión básica y primer dataframe

In [2]:
import requests
import pandas as pd

url = "https://swapi.dev/api/vehicles"

response = requests.get(url)

if response.status_code == 200:
    print ("Éxito conexión establecida")
    payload = response.json()
    data_vehiculos = payload['results']

    df_vehicles = pd.DataFrame(data_vehiculos)

    print("Primeras 5 filas:")
    print(df_vehicles.head())
    print("\nColumnas obtenidas:")
    print(df_vehicles.columns.tolist())

else:
    print(f"Error en la conexión. Código: {response.status_code}")

Éxito conexión establecida
Primeras 5 filas:
                 name                           model  \
0        Sand Crawler                  Digger Crawler   
1      T-16 skyhopper                  T-16 skyhopper   
2    X-34 landspeeder                X-34 landspeeder   
3  TIE/LN starfighter  Twin Ion Engine/Ln Starfighter   
4         Snowspeeder                 t-47 airspeeder   

                  manufacturer cost_in_credits length max_atmosphering_speed  \
0  Corellia Mining Corporation          150000  36.8                      30   
1            Incom Corporation           14500  10.4                    1200   
2         SoroSuub Corporation           10550   3.4                     250   
3         Sienar Fleet Systems         unknown    6.4                   1200   
4            Incom corporation         unknown    4.5                    650   

  crew passengers cargo_capacity consumables  vehicle_class  \
0   46         30          50000    2 months        wheeled   
1    

## 2.- Gestión de paginación

In [3]:
url_people = "https://swapi.dev/api/people/"
all_people = []

while url_people:
    response = requests.get(url_people)
    data = response.json()
    
    all_people.extend(data['results'])
    
    url_people = data.get('next')

df_people = pd.DataFrame(all_people)

api_total_count = data.get('count')
print(f"Registros en el DataFrame: {len(df_people)}")
print(f"Total esperado por la API: {api_total_count}")
df_people.head(5)

Registros en el DataFrame: 82
Total esperado por la API: 82


,name,height,mass,hair_color,skin_color,eye_color,birth_year,gender,homeworld,films,species,vehicles,starships,created,edited,url
0,Luke Skywalker,172,77,blond,fair,blue,19BBY,male,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[],"[https://swapi.dev/api/vehicles/14/, https://s...","[https://swapi.dev/api/starships/12/, https://...",2014-12-09T13:50:51.644000Z,2014-12-20T21:17:56.891000Z,https://swapi.dev/api/people/1/
1,C-3PO,167,75,n/a,gold,yellow,112BBY,n/a,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[https://swapi.dev/api/species/2/],[],[],2014-12-10T15:10:51.357000Z,2014-12-20T21:17:50.309000Z,https://swapi.dev/api/people/2/
2,R2-D2,96,32,n/a,"white, blue",red,33BBY,n/a,https://swapi.dev/api/planets/8/,"[https://swapi.dev/api/films/1/, https://swapi...",[https://swapi.dev/api/species/2/],[],[],2014-12-10T15:11:50.376000Z,2014-12-20T21:17:50.311000Z,https://swapi.dev/api/people/3/
3,Darth Vader,202,136,none,white,yellow,41.9BBY,male,https://swapi.dev/api/planets/1/,"[https://swapi.dev/api/films/1/, https://swapi...",[],[],[https://swapi.dev/api/starships/13/],2014-12-10T15:18:20.704000Z,2014-12-20T21:17:50.313000Z,https://swapi.dev/api/people/4/
4,Leia Organa,150,49,brown,light,brown,19BBY,female,https://swapi.dev/api/planets/2/,"[https://swapi.dev/api/films/1/, https://swapi...",[],[https://swapi.dev/api/vehicles/30/],[],2014-12-10T15:20:09.791000Z,2014-12-20T21:17:50.315000Z,https://swapi.dev/api/people/5/


## 3.- Cruce de datos

In [4]:
def get_planet_info(url):
    try:
        res = requests.get(url)
        p_data = res.json()
        return p_data['name'], p_data['terrain'], p_data['population']
    except:
        return None, None, None

df_sample = df_people.iloc[:20].copy()

df_sample[['planet_name', 'planet_terrain', 'planet_population']] = df_sample['homeworld'].apply(
    lambda x: pd.Series(get_planet_info(x))
)

print(df_sample[['name', 'planet_name', 'planet_terrain', 'planet_population']].head())

             name planet_name                            planet_terrain  \
0  Luke Skywalker    Tatooine                                    desert   
1           C-3PO    Tatooine                                    desert   
2           R2-D2       Naboo  grassy hills, swamps, forests, mountains   
3     Darth Vader    Tatooine                                    desert   
4     Leia Organa    Alderaan                     grasslands, mountains   

  planet_population  
0            200000  
1            200000  
2        4500000000  
3            200000  
4        2000000000  


## 4.- Expansión de filas (opcional)

In [5]:
df_expanded = df_sample.explode('films')

film_cache = {}

def get_film_title(url):
    if url not in film_cache:
        res = requests.get(url)
        film_cache[url] = res.json()['title']
    return film_cache[url]

df_expanded['films'] = df_expanded['films'].apply(get_film_title)

print("\nDataframe expandido y títulos de películas:")
print(df_expanded[['name', 'films']].head(20))


Dataframe expandido y títulos de películas:
             name                    films
0  Luke Skywalker               A New Hope
0  Luke Skywalker  The Empire Strikes Back
0  Luke Skywalker       Return of the Jedi
0  Luke Skywalker      Revenge of the Sith
1           C-3PO               A New Hope
1           C-3PO  The Empire Strikes Back
1           C-3PO       Return of the Jedi
1           C-3PO       The Phantom Menace
1           C-3PO     Attack of the Clones
1           C-3PO      Revenge of the Sith
2           R2-D2               A New Hope
2           R2-D2  The Empire Strikes Back
2           R2-D2       Return of the Jedi
2           R2-D2       The Phantom Menace
2           R2-D2     Attack of the Clones
2           R2-D2      Revenge of the Sith
3     Darth Vader               A New Hope
3     Darth Vader  The Empire Strikes Back
3     Darth Vader       Return of the Jedi
3     Darth Vader      Revenge of the Sith
